# 05 · Workflows: orquestar las tareas

⏱ **18 min** · Módulo 5 · Slide de la Intro: 21 (Workflows: orquestar tareas)

**Objetivo:** convertir el ETL del notebook 04 en un **Job** de cuatro tareas que corre solo, con dependencias, parámetros,
reintentos, horario y alertas.

```
Disparador: horario · archivo nuevo · ejecución manual
        │
     ingesta ──► transformacion ──► validacion ──► publicacion
   RAW y bronze      silver           reglas           gold
```

| Paso | Qué haces |
|---|---|
| 1 | Las cuatro tareas: un notebook pequeño por tarea |
| 2 | Definir el Job |
| 3 | Crearlo |
| 4 | Ejecutarlo con parámetros |
| 5 | Una validación que falla: la publicación no corre |
| 6 | Los disparadores |
| 7 | Hazlo tú en la interfaz |

> En la interfaz, Workflows está en el menú **Jobs & Pipelines**.

**Requiere:** notebook 04.

In [0]:
%run ./00_Setup

## Paso 1 · Las cuatro tareas

Cada tarea es un notebook pequeño que hace **una sola cosa**. Están en la carpeta `tareas`.

In [0]:
LAB_DIR = os.getcwd()        # la carpeta de este laboratorio en el Workspace
print(LAB_DIR)
print(sorted(os.listdir(f"{LAB_DIR}/tareas")))

| Tarea | Notebook | Qué hace | Parámetros |
|---|---|---|---|
| `ingesta` | `tareas/1_ingesta` | API → RAW → bronze | `start_date`, `dias` |
| `transformacion` | `tareas/2_transformacion` | bronze → silver | — |
| `validacion` | `tareas/3_validacion` | Revisa silver; si una regla falla, lanza un error | `min_filas` |
| `publicacion` | `tareas/4_publicacion` | silver → gold | — |

👀 **Ábrelos:** tienen dos o tres celdas y usan las funciones del notebook 04. Los parámetros se leen con *widgets*.

## Paso 2 · Definir el Job

Un Job se describe con sus tareas y sus reglas. Ubica en el código las cuatro capacidades del slide 21.

In [0]:
job = {
    "name": JOB_NAME,
    "parameters": [                                              # PARÁMETROS por ejecución
        {"name": "start_date", "default": "2024-03-01"},
        {"name": "dias",       "default": "3"},
        {"name": "min_filas",  "default": "1"},
    ],
    "tasks": [
        {"task_key": "ingesta",
         "notebook_task": {"notebook_path": f"{LAB_DIR}/tareas/1_ingesta"},
         "max_retries": 1, "min_retry_interval_millis": 30000},  # REINTENTO automático: la API puede fallar
        {"task_key": "transformacion",
         "depends_on": [{"task_key": "ingesta"}],                # DEPENDENCIA: corre cuando termina ingesta
         "notebook_task": {"notebook_path": f"{LAB_DIR}/tareas/2_transformacion"}},
        {"task_key": "validacion",
         "depends_on": [{"task_key": "transformacion"}],
         "notebook_task": {"notebook_path": f"{LAB_DIR}/tareas/3_validacion"},
         "disable_auto_optimization": True},                     # sin reintentos: repetir no arregla datos malos
        {"task_key": "publicacion",
         "depends_on": [{"task_key": "validacion"}],
         "notebook_task": {"notebook_path": f"{LAB_DIR}/tareas/4_publicacion"}},
    ],
    "schedule": {                                                # DISPARADOR: todos los días a las 6:00 (en pausa)
        "quartz_cron_expression": "0 0 6 * * ?",
        "timezone_id": "America/Lima",
        "pause_status": "PAUSED",
    },
    "email_notifications": {"on_failure": [USER]},               # ALERTA: un correo si el Job falla
}

## Paso 3 · Crearlo

In [0]:
JOB_ID = create_job(job)

👀 **Abre el enlace:** verás el gráfico con las cuatro tareas encadenadas y, a la derecha, el horario (en pausa), los parámetros
y las notificaciones. No definimos ningún cluster: en la Free Edition las tareas corren en cómputo **serverless**.

## Paso 4 · Ejecutarlo con parámetros

Llegan dos días más: lanzamos el Job con otra fecha, sin tocar el código. Tarda unos minutos: mientras tanto, abre el enlace.

In [0]:
run_job(JOB_ID, start_date="2024-03-04", dias="2")

In [0]:
%sql
SELECT airdate, sum(episodes) AS episodios
FROM gold_network_daily
GROUP BY airdate
ORDER BY airdate

👀 **Qué observar:** en la interfaz las tareas se ponen verdes una por una: cada una espera a la anterior.
Los parámetros llegaron a la tarea `ingesta` y gold ahora tiene cinco días.

## Paso 5 · Una validación que falla

Pedimos un mínimo de filas imposible. ¿Qué pasa con la publicación?

In [0]:
run_job(JOB_ID, min_filas="1000000")

In [0]:
%sql
SELECT airdate, sum(episodes) AS episodios
FROM gold_network_daily
GROUP BY airdate
ORDER BY airdate

👀 **Qué observar:** `validacion` falló y `publicacion` ni siquiera empezó (`UPSTREAM_FAILED`): gold sigue igual que antes,
nada malo llegó al negocio. Además te llegó un correo. En la interfaz, el botón **Repair run** vuelve a correr solo lo que falló.

## Paso 6 · Los disparadores

| Disparador | Cómo se define | Cuándo usarlo |
|---|---|---|
| Manual | Botón **Run now** o `run_job()` | Pruebas y reprocesos |
| Horario | `"schedule"` con una expresión cron | Cargas diarias |
| Archivo nuevo | `"trigger": {"file_arrival": {"url": "/Volumes/.../raw/schedule/"}}` | Cuando no sabes a qué hora llega el dato |

Nuestro Job tiene el horario **en pausa** para no gastar cómputo. En un proyecto real lo activas en la interfaz (**Schedules & Triggers → Resume**).

## Paso 7 · Hazlo tú en la interfaz

1. **Jobs & Pipelines → Create → Job**.
2. Agrega una tarea de tipo **Notebook** y elige `tareas/1_ingesta`.
3. Agrega otra con `tareas/2_transformacion` y en **Depends on** elige la primera.
4. Pulsa **Run now** y mira el gráfico. Al terminar, bórralo.

Es lo mismo que hizo `create_job(job)`: el botón **⋮ → View as code** de cualquier Job te muestra ese diccionario.
En el módulo 7 (Asset Bundles) lo escribirás en un archivo YAML dentro de Git.

## ✔ Checkpoint

- [ ] Creé un Job de cuatro tareas con dependencias
- [ ] Lo ejecuté con parámetros distintos a los de por defecto
- [ ] Vi que una tarea fallida detiene a las que dependen de ella
- [ ] Ubico en la definición los reintentos, el horario y la alerta

**Siguiente:** `06_Pipeline_Declarativo`